In [1]:
!git clone https://github.com/echanatwell/LLM_weight_quantization_triton.git

Cloning into 'LLM_weight_quantization_triton'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 37 (delta 10), reused 34 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 15.43 KiB | 687.00 KiB/s, done.
Resolving deltas: 100% (10/10), done.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import triton
import triton.language as tl

import time
import os
os.chdir('/kaggle/working/LLM_weight_quantization_triton')
from tqdm import tqdm

from CustomLayers import DummyLinear, QuantizedLinearGlobalTorch
from benchmark.perplexity import measure_ppl

In [28]:
def change_linear_layer(model, new_layer, device):
    for layer in model.model.layers:
        #layer.self_attn.q_proj = new_layer(layer.self_attn.q_proj, device)
        #layer.self_attn.k_proj = new_layer(layer.self_attn.k_proj, device)
        #layer.self_attn.v_proj = new_layer(layer.self_attn.v_proj, device)
        #layer.self_attn.o_proj = new_layer(layer.self_attn.o_proj, device)

        layer.mlp.gate_proj = new_layer(layer.mlp.gate_proj, device)
        layer.mlp.up_proj = new_layer(layer.mlp.up_proj, device)
        layer.mlp.down_proj = new_layer(layer.mlp.down_proj, device)

    #model.lm_head = new_layer(model.lm_head, device)
    torch.cuda.empty_cache()

    return model


def calc_model_size(model):
    param_mem = 0.
    buffer_mem = 0.
    for param in model.parameters():
        param_mem += param.nelement() * param.element_size()
    for buffer in model.buffers():
        buffer_mem += buffer.nelement() * buffer.element_size()

    return (param_mem + buffer_mem) / (2 ** 30)


def time_inference(model, tokenizer, device, text, max_length=50):

    inputs = tokenizer(text, return_tensors='pt').to(device)
    
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=max_length, 
                               pad_token_id=tokenizer.eos_token_id)
    end_time = time.time()
    
    inference_time = end_time - start_time
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return inference_time, generated_text


def verify_layer_accuracy(orig_model, quant_model, tokenizer, text='hello'):
    inputs = tokenizer(text, return_tensors='pt').to(orig_model.device)
    inputs_ = tokenizer(text, return_tensors='pt').to(quant_model.device)

    with torch.no_grad():
        orig_outputs = orig_model(**inputs, output_hidden_states=True)

    with torch.no_grad():
        quant_outputs = quant_model(**inputs_, output_hidden_states=True)

    for i, (orig_hidden, quant_hidden) in enumerate(zip(orig_outputs.hidden_states, quant_outputs.hidden_states)):
        print(orig_hidden.shape, quant_hidden.shape)
        mse = F.mse_loss(orig_hidden.cpu(), quant_hidden.cpu())
        cos_sim = F.cosine_similarity(orig_hidden.flatten().cpu(), quant_hidden.flatten().cpu(), dim=0)
        print(f'layer {i}: mse = {mse:.6f}, cosine_sim = {cos_sim:.6f}')

In [21]:
raw_datasets = load_dataset('zhengxuanzenwu/wikitext-2-split-128', split='test')

Repo card metadata block was not found. Setting CardData to empty.


In [5]:
prompts = [x['text'] for x in raw_datasets if len(x['text']) > 0]
print('Number of sequences:', len(prompts))

Number of sequences: 8192


In [6]:
model_id = 'unsloth/Llama-3.2-1B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32, device_map='cuda:0')

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

...

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [34]:
# del model
# del custom_model
# torch.cuda.empty_cache()

In [8]:
custom_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32, device_map='cuda:1')
custom_model = change_linear_layer(custom_model, DummyLinear, custom_model.device)

In [9]:
print(model.model.layers[0].self_attn.q_proj.weight.dtype)
(custom_model.model.layers[0].self_attn.q_proj.weight.dtype)

torch.float32


torch.float32

In [10]:
orig_ppl, orig_time = measure_ppl(prompts, model, tokenizer)

100%|██████████| 8192/8192 [08:13<00:00, 16.58it/s]



Perplexity: 345.0570
Mean time per sample: 0.060 s


In [11]:
custom_ppl, custom_time = measure_ppl(prompts, custom_model, tokenizer)

100%|██████████| 8192/8192 [07:48<00:00, 17.48it/s]


Perplexity: 345.0570
Mean time per sample: 0.057 s


In [8]:
x = nn.Linear(1000, 1000, bias=True, dtype=torch.float32).to('cuda:1')
y = torch.randn(1000, 1000, dtype=torch.float32, requires_grad=False).to('cuda:1')
out_real = x(y)
l = QuantizedLinearGlobalTorch(x, 'cuda:1')
out_quant = l(y)

print(F.l1_loss(out_real, out_quant))

del x
del y
del l
torch.cuda.empty_cache()

tensor(0.0329, device='cuda:1', grad_fn=<MeanBackward0>)


In [29]:
custom_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32, device_map='cuda:1')
custom_model = change_linear_layer(custom_model, QuantizedLinearGlobalTorch, custom_model.device)

In [30]:
custom_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): QuantizedLinearGlobalTorch()
          (up_proj): QuantizedLinearGlobalTorch()
          (down_proj): QuantizedLinearGlobalTorch()
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb): LlamaRotaryEmbedding()
  )
  (lm_head): Linear(in_features=2

In [31]:
text = 'Hello how is the weather?'
orig_model_size = calc_model_size(model)
orig_inf_time, orig_inf_text = time_inference(model, tokenizer, model.device, text)
print('%f GB; %f s.;\n%s' % (orig_model_size, orig_inf_time, orig_inf_text))

4.603768 GB; 1.075579 s.;
Hello how is the weather? I'm looking for a place to get some fresh air and enjoy the outdoors. I'm in the area of Los Angeles, California.

There are plenty of options for you to choose from, whether you're looking for


In [32]:
text = 'Hello how is the weather?'
quant_model_size = calc_model_size(custom_model)
quant_inf_time, quant_inf_text = time_inference(custom_model, tokenizer, custom_model.device, text)
print('%f GB; %f s.;\n%s' % (quant_model_size, quant_inf_time, quant_inf_text))

1.978768 GB; 8.088783 s.;
Hello how is the weather?Leading leading leading leading leading leadingleadleadLeadLead lead lead lead LeadLead lead lead lead lead lead lead lead lead lead Lead lead lead lead lead lead lead lead lead lead lead lead lead contact contact contact contact contact contact


In [33]:
verify_layer_accuracy(model, custom_model, tokenizer)

torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 0: mse = 0.000000, cosine_sim = 1.000000
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 1: mse = 0.005562, cosine_sim = 0.993758
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 2: mse = 9.103346, cosine_sim = 0.984845
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 3: mse = 9.147719, cosine_sim = 0.984810
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 4: mse = 9.176298, cosine_sim = 0.984829
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 5: mse = 9.219006, cosine_sim = 0.984871
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 6: mse = 9.254936, cosine_sim = 0.984943
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 7: mse = 9.293110, cosine_sim = 0.985044
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 8: mse = 9.339248, cosine_sim = 0.985086
torch.Size([1, 2, 2048]) torch.Size([1, 2, 2048])
layer 9: mse = 9.376293, cosine_sim = 0.985096
torch.Size([1, 2, 2048]) torch